# Lesson 07 - Padrão de Design de Planejamento

Este notebook demonstra o **Padrão de Design de Planejamento** para agentes de IA usando o Microsoft Agent Framework.  
Você aprenderá como dividir solicitações de viagem complexas em subtarefas estruturadas, atribuí-las a agentes especialistas  
e executar o plano resultante — tudo usando saída estruturada com modelos Pydantic.


## Configuração


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Decomposição da Tarefa

A decomposição da tarefa é o cerne do padrão de design de planejamento. Em vez de pedir a um único agente para lidar com uma solicitação complexa de ponta a ponta, dividimos o problema em **subtarefas** menores e bem definidas. Cada subtarefa é atribuída a um agente especialista (por exemplo, voos, hotéis, atividades) com prioridades claras e ordenação de dependência.

Essa abordagem oferece vários benefícios:
- **Clareza**: cada subtarefa tem uma responsabilidade única.
- **Paralelismo**: subtarefas independentes podem ser executadas simultaneamente.
- **Confiabilidade**: falhas são isoladas em subtarefas individuais.
- **Rastreamento do orçamento**: os custos são estimados por subtarefa e acumulados.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Criando um Agente de Planejamento com Saída Estruturada

O agente de planejamento atua como um **coordenador da recepção**. Dada uma solicitação de viagem de alto nível, ele
produz um `TravelPlan` estruturado — decompondo a solicitação em subtarefas, definindo prioridades,
e identificando dependências para que um concierge ou camada de execução possa realizar o trabalho.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Executando um Plano com Ferramentas Especializadas

Uma vez que o agente da recepção tenha produzido um plano estruturado, o **agente concierge** o executa.
Cada ferramenta especializada lida com uma categoria de subtarefa (voos, hotéis, atividades). O concierge
itera através das subtarefas do plano na ordem de dependência e despacha cada uma para a
ferramenta apropriada.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Resumo

Nesta lição você aprendeu o **Padrão de Projeto de Planejamento** para agentes de IA:

1. **Decomposição de Tarefas** — Um agente de planejamento da recepção divide uma solicitação complexa de viagem em subtarefas estruturadas usando modelos Pydantic, atribuindo cada uma a um agente especialista com prioridades e dependências.
2. **Saída Estruturada** — Ao passar um `response_format`, o agente retorna um objeto `TravelPlan` validado em vez de texto livre, tornando o processamento posterior mais confiável.
3. **Execução do Plano** — Um agente concierge itera pelas subtarefas usando ferramentas especializadas (`book_flight`, `reserve_hotel`, `book_activity`) para realizar o plano e relatar os resultados.

Esse padrão separa *o que fazer* (planejamento) de *como fazer* (execução), tornando os agentes mais modulares, testáveis e fáceis de estender.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:  
Este documento foi traduzido utilizando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos para garantir a precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte oficial. Para informações críticas, recomenda-se a tradução profissional feita por humanos. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
